## Data Loading

In [5]:
# Proof-of-concept: scrape one confederation WCQ stats page directly.
import pandas as pd

URL = "https://fbref.com/en/comps/6/stats/WCQ----UEFA-M-Stats"
CACHE = CACHE_DIR / "wcq_uefa_2026.html"

reader = fbref.get(URL, CACHE)
tree = html.parse(reader)
parser = etree.HTMLParser(recover=True)

# The main player stats table on a comp's /stats/ page is usually `stats_standard`
# (no comp-id suffix because the page IS that comp). FBref often comments it out.
candidates = []
for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
    root = etree.fromstring(f"<root>{c.text}</root>", parser)
    candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))

# Dedupe by id
seen = set()
unique = []
for t in candidates:
    if t.get("id") in seen:
        continue
    seen.add(t.get("id"))
    unique.append(t)

print(f"Found {len(unique)} stats_standard table(s): {[t.get('id') for t in unique]}")

# Parse the first one
df = _parse_table(unique[0])
print(f"\nShape: {df.shape}")
print(f"\nTop-level column groups: {df.columns.get_level_values(0).unique().tolist()}")
print(f"\nLeaf column names: {df.columns.get_level_values(1).tolist()}")

# Inspect — what does the Squad/Nation column look like?
print(f"\nFirst 3 rows:")
print(df.head(3))

# Flatten so we can poke at it
df_flat = df.copy()
df_flat.columns = [b if (not a or a == b or a.startswith("Unnamed")) else f"{a}_{b}" for a, b in df_flat.columns]
df_flat = df_flat.reset_index(drop=True)

# Find the team-affiliation column
team_cols = [c for c in df_flat.columns if c.lower() in ("squad", "team", "nation")]
print(f"\nTeam-affiliation columns: {team_cols}")

if team_cols:
    tc = team_cols[0]
    print(f"\nUnique {tc} values ({df_flat[tc].nunique()}):")
    print(sorted(df_flat[tc].dropna().unique().tolist()))

print(f"\nTotal players: {len(df_flat)}")

Found 1 stats_standard table(s): ['stats_standard']

Shape: (1590, 24)

Top-level column groups: ['Unnamed: 0_level_0', 'Unnamed: 1_level_0', 'Unnamed: 2_level_0', 'Unnamed: 3_level_0', 'Unnamed: 4_level_0', 'Unnamed: 5_level_0', 'Playing Time', 'Performance', 'Per 90 Minutes', 'Unnamed: 23_level_0']

Leaf column names: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls', 'Ast', 'G+A', 'G-PK', 'G+A-PK', 'Matches']

First 3 rows:
  Unnamed: 0_level_0  Unnamed: 1_level_0 Unnamed: 2_level_0  \
                  Rk              Player                Pos   
0                  1  Bárður Á Reynatrøð                 GK   
1                  2      Thelo Aasgaard                 MF   
2                  3          Liel Abada                 DF   

  Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Playing Time  \
               Squad                Age               Born           MP   
0      Fa

In [6]:
# Find URLs for all other "FIFA World Cup Qualification" comps from FBref's master comps index.
leagues_html = Path.home() / "soccerdata" / "data" / "FBref" / "leagues.html"
tree = html.parse(str(leagues_html))

# Find all anchor tags with text matching "World Cup Qualification"
wcq_links = []
for a in tree.xpath("//a"):
    text = (a.text or "").strip()
    href = a.get("href", "")
    if "World Cup Qualification" in text and "Women" not in text:
        wcq_links.append((text, href))

print(f"Found {len(wcq_links)} WCQ links:")
for text, href in wcq_links:
    print(f"  {text:55s} → {href}")


Found 7 WCQ links:
  FIFA World Cup Qualification — Inter-confederation play-offs → /en/comps/255/history/FIFA-World-Cup-Qualification----Inter-confederation-play-offs-Seasons
  FIFA World Cup Qualification — CAF                      → /en/comps/2/history/WCQ----CAF-M-Seasons
  FIFA World Cup Qualification — CONCACAF                 → /en/comps/3/history/WCQ----CONCACAF-M-Seasons
  FIFA World Cup Qualification — CONMEBOL                 → /en/comps/4/history/WCQ----CONMEBOL-M-Seasons
  FIFA World Cup Qualification — OFC                      → /en/comps/5/history/WCQ----OFC-M-Seasons
  FIFA World Cup Qualification — UEFA                     → /en/comps/6/history/WCQ----UEFA-M-Seasons
  FIFA World Cup Qualification — AFC                      → /en/comps/7/history/WCQ----AFC-M-Seasons


In [7]:
# Scrape all 7 WCQ pages → one combined player-stats DataFrame for the 2026 cycle.
import re

CONFED_URLS = {
    "UEFA":      "https://fbref.com/en/comps/6/stats/WCQ----UEFA-M-Stats",
    "CAF":       "https://fbref.com/en/comps/2/stats/WCQ----CAF-M-Stats",
    "CONCACAF":  "https://fbref.com/en/comps/3/stats/WCQ----CONCACAF-M-Stats",
    "CONMEBOL":  "https://fbref.com/en/comps/4/stats/WCQ----CONMEBOL-M-Stats",
    "OFC":       "https://fbref.com/en/comps/5/stats/WCQ----OFC-M-Stats",
    "AFC":       "https://fbref.com/en/comps/7/stats/WCQ----AFC-M-Stats",
    "InterConf": "https://fbref.com/en/comps/255/stats/FIFA-World-Cup-Qualification----Inter-confederation-play-offs-Stats",
}

def fetch_and_parse_wcq(confed, url):
    cache = CACHE_DIR / f"wcq_{confed}_2026.html"
    reader = fbref.get(url, cache)
    tree = html.parse(reader)
    parser = etree.HTMLParser(recover=True)

    candidates = []
    for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
        root = etree.fromstring(f"<root>{c.text}</root>", parser)
        candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
    candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))

    seen, unique = set(), []
    for t in candidates:
        if t.get("id") in seen:
            continue
        seen.add(t.get("id"))
        unique.append(t)
    if not unique:
        return None
    df = _parse_table(unique[0])
    # Flatten MultiIndex columns
    df.columns = [b if (not a or a == b or str(a).startswith("Unnamed")) else f"{a}_{b}"
                  for a, b in df.columns]
    df = df.reset_index(drop=True)
    df["confederation"] = confed
    return df

frames = []
for confed, url in CONFED_URLS.items():
    df = fetch_and_parse_wcq(confed, url)
    if df is None:
        print(f"  {confed}: NO TABLE FOUND")
        continue
    print(f"  {confed}: {len(df)} player rows, {df['Squad'].nunique()} squads")
    frames.append(df)

wcq_all = pd.concat(frames, ignore_index=True)
print(f"\n=== Combined: {len(wcq_all)} player rows across {wcq_all['Squad'].nunique()} nations ===")
print(f"Columns: {wcq_all.columns.tolist()}")

# Cross-check coverage against our WC squads
wc_players = pd.read_csv("../data/processed/player_fixtures.csv")[["player", "team"]].drop_duplicates()
wc_nations = sorted(wc_players["team"].unique())
print(f"\nWC nations in our roster: {len(wc_nations)}")

wcq_nations = set(wcq_all["Squad"].dropna().unique())
present     = [n for n in wc_nations if n in wcq_nations]
missing     = [n for n in wc_nations if n not in wcq_nations]
print(f"Present in WCQ scrape: {len(present)} / {len(wc_nations)}")
print(f"Missing (likely name-mismatch — fix with a mapping): {missing}")

  UEFA: 1590 player rows, 54 squads
  CAF: 2100 player rows, 53 squads
  CONCACAF: 952 player rows, 32 squads
  CONMEBOL: 462 player rows, 10 squads
  OFC: 227 player rows, 11 squads
  AFC: 1517 player rows, 46 squads
  InterConf: NO TABLE FOUND

=== Combined: 6848 player rows across 206 nations ===
Columns: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A', 'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G-PK', 'Per 90 Minutes_G+A-PK', 'Matches', 'confederation']

WC nations in our roster: 48
Present in WCQ scrape: 43 / 48
Missing (likely name-mismatch — fix with a mapping): ['Bosnia and Herzegovina', 'Cabo Verde', 'Canada', 'Mexico', 'USA']


In [8]:
# Verify Cape Verde / Cabo Verde mapping, then build the filtered international-stats table.

# Sanity: list all FBref Squad names that look like Cape Verde
print("Squad names containing 'verde' or 'cape':")
for s in sorted(wcq_all["Squad"].dropna().unique()):
    if "verde" in s.lower() or "cape" in s.lower():
        print(f"  {s}")

# Map our WC squad names → FBref Squad names. Only entries that actually differ.
WC_TO_FBREF_SQUAD = {
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Cabo Verde": "Cape Verde",
    # Add more here if other mismatches surface
}

# Apply mapping to our roster, filter wcq_all to our 48
wc_players["fbref_squad"] = wc_players["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_players["team"])
wc_squads_fbref = set(wc_players["fbref_squad"].unique())

intl = wcq_all[wcq_all["Squad"].isin(wc_squads_fbref)].copy()

print(f"\nFiltered international-stats rows: {len(intl)}")
print(f"Squads represented: {intl['Squad'].nunique()} / 48")

# Per-team match/minute summary
summary = (
    intl.assign(min_=pd.to_numeric(intl["Playing Time_Min"].astype(str).str.replace(",", ""), errors="coerce"))
        .groupby("Squad")
        .agg(players=("Player", "nunique"),
             total_min=("min_", "sum"),
             max_mp=("Playing Time_MP", "max"))
        .sort_values("max_mp", ascending=False)
)
print(f"\nPer-team summary (top 10 by matches played):")
print(summary.head(10))
print(f"\nPer-team summary (bottom 10 by matches played):")
print(summary.tail(10))

# Which of the 48 still have zero rows (should be USA, Canada, Mexico)
covered = set(intl["Squad"].unique())
still_missing = [t for t in wc_squads_fbref if t not in covered]
print(f"\nStill missing (expected to be the 3 co-hosts): {still_missing}")

Squad names containing 'verde' or 'cape':
  Cape Verde

Filtered international-stats rows: 1653
Squads represented: 45 / 48

Per-team summary (top 10 by matches played):
                players  total_min  max_mp
Squad                                     
Iraq                 48      19650      20
Ecuador              44      17706      18
Brazil               60      17810      18
Colombia             43      17816      18
Argentina            35      17738      18
Uruguay              41      17820      17
Paraguay             46      17773      17
Qatar                56      17782      16
Korea Republic       52      15840      16
Jordan               37      15840      16

Per-team summary (bottom 10 by matches played):
             players  total_min  max_mp
Squad                                  
Austria           28       7919       8
Croatia           32       7920       8
England           32       7920       8
Switzerland       23       5940       6
France            30     

In [9]:
out = "../data/processed/international_wcq_2026.csv"
intl.to_csv(out, index=False)
print(f"Wrote {len(intl)} rows → {out}")

Wrote 1653 rows → ../data/processed/international_wcq_2026.csv


## More Data!! 

We want to get more international football player level data, e.g. Copa America for South American nations, UEFA Nations League for UEFA nations....

In [10]:
# Discover URLs for the 5 v2 comps from FBref's master comps index.
search_terms = [
    "UEFA Nations League",
    "Gold Cup",
    "Africa Cup of Nations",
    "UEFA Euro",
    "Copa Am",
]

leagues_html = Path.home() / "soccerdata" / "data" / "FBref" / "leagues.html"
tree = html.parse(str(leagues_html))

print("Candidate links per search term:\n")
for term in search_terms:
    print(f"=== {term!r} ===")
    matches = []
    for a in tree.xpath("//a"):
        text = (a.text or "").strip()
        href = a.get("href", "")
        if term.lower() in text.lower() and "Women" not in text and "U-" not in text and "Youth" not in text:
            matches.append((text, href))
    if not matches:
        print("  (no matches)")
    for text, href in matches:
        print(f"  {text:55s} → {href}")
    print()


Candidate links per search term:

=== 'UEFA Nations League' ===
  UEFA Nations League                                     → /en/comps/677/UEFA-Nations-League-Stats
  UEFA Nations League                                     → /en/comps/677/history/UEFA-Nations-League-Seasons
  UEFA Nations League                                     → /en/comps/677/UEFA-Nations-League-Stats

=== 'Gold Cup' ===
  CONCACAF Gold Cup                                       → /en/comps/681/history/Gold-Cup-Seasons
  CONCACAF Gold Cup                                       → /en/comps/681/Gold-Cup-Stats

=== 'Africa Cup of Nations' ===
  Africa Cup of Nations                                   → /en/comps/656/history/Africa-Cup-of-Nations-Seasons
  Africa Cup of Nations qualification                     → /en/comps/657/history/Africa-Cup-of-Nations-qualification-Seasons
  Africa Cup of Nations                                   → /en/comps/656/Africa-Cup-of-Nations-Stats

=== 'UEFA Euro' ===
  UEFA European Football

In [ ]:
# for _, comp, _ in V2_COMPS:
#     cache_file = CACHE_DIR / f"{comp.replace(' ', '_')}.html"
#     if cache_file.exists():
#         cache_file.unlink()
#         print(f"Deleted {cache_file.name}")

Deleted UEFA_Nations_League.html
Deleted AFCON_2025.html
Deleted Gold_Cup_2025.html
Deleted Euro_2024.html
Deleted Copa_America_2024.html


In [15]:
# v2: add 5 more international competitions, with a `competition` column for downstream filtering.

V2_COMPS = [
    ("UEFA",     "UEFA Nations League",   "https://fbref.com/en/comps/677/stats/UEFA-Nations-League-Stats"),
    ("CAF",      "AFCON 2025",            "https://fbref.com/en/comps/656/stats/Africa-Cup-of-Nations-Stats"),
    ("CONCACAF", "Gold Cup 2025",         "https://fbref.com/en/comps/681/stats/Gold-Cup-Stats"),
    ("UEFA",     "Euro 2024",             "https://fbref.com/en/comps/676/stats/UEFA-Euro-Stats"),
    ("CONMEBOL", "Copa America 2024",     "https://fbref.com/en/comps/685/stats/Copa-America-Stats"),
]

def fetch_comp_standard(url, cache_name):
    cache = CACHE_DIR / cache_name
    reader = fbref.get(url, cache)
    tree = html.parse(reader)
    parser = etree.HTMLParser(recover=True)

    candidates = []
    for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
        root = etree.fromstring(f"<root>{c.text}</root>", parser)
        candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
    candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))
    seen, unique = set(), []
    for t in candidates:
        if t.get("id") in seen:
            continue
        seen.add(t.get("id"))
        unique.append(t)
    if not unique:
        return None, None

    tbl = unique[0]
    caption_el = tbl.xpath(".//caption")
    caption = "".join(caption_el[0].itertext()).strip() if caption_el else tbl.get("id", "")

    df = _parse_table(tbl)
    df.columns = [b if (not a or a == b or str(a).startswith("Unnamed")) else f"{a}_{b}"
                  for a, b in df.columns]
    df = df.reset_index(drop=True)
    return caption, df

v2_frames = []
for confed, comp, url in V2_COMPS:
    cache_name = f"{comp.replace(' ', '_')}.html"
    caption, df = fetch_comp_standard(url, cache_name)
    if df is None:
        print(f"  {comp}: NO TABLE FOUND ({url})")
        continue
    df["confederation"] = confed
    df["competition"] = comp
    print(f"  {comp:25s} | caption: {caption!r}")
    print(f"  {'':25s}   {len(df)} player rows, {df['Squad'].nunique()} squads")
    v2_frames.append(df)

v2_all = pd.concat(v2_frames, ignore_index=True)

# Add `competition` column to v1 (intl) so schemas align
intl["competition"] = intl["confederation"].map({
    "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
    "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
})

# Filter v2 to our 48 WC squads (reuse the mapping from v1)
v2_filtered = v2_all[v2_all["Squad"].isin(wc_squads_fbref)].copy()
print(f"\nv2 filtered to WC squads: {len(v2_filtered)} rows across {v2_filtered['Squad'].nunique()} squads")

# Stack v1 + v2
intl_all = pd.concat([intl, v2_filtered], ignore_index=True)
print(f"\n=== Combined v1+v2: {len(intl_all)} rows, {intl_all['Squad'].nunique()} squads ===")
print(intl_all.groupby("competition").size().sort_values(ascending=False))

# Co-host check — did we pick up USA/Canada/Mexico?
cohost_data = intl_all[intl_all["Squad"].isin(["United States", "USA", "Canada", "Mexico"])]
print(f"\nCo-host coverage now:")
print(cohost_data.groupby(["Squad", "competition"]).size())

  UEFA Nations League       | caption: 'Player Standard Stats 2024-2025 UEFA Nations League Table'
                              1457 player rows, 54 squads
  AFCON 2025                | caption: 'Player Standard Stats 2025 Africa Cup of Nations Table'
                              543 player rows, 24 squads
  Gold Cup 2025             | caption: 'Player Standard Stats 2025 Gold Cup Table'
                              335 player rows, 16 squads
  Euro 2024                 | caption: 'Player Standard Stats 2024 UEFA Euro 2024 Table'
                              493 player rows, 24 squads
  Copa America 2024         | caption: 'Player Standard Stats 2024 Copa América Table'
                              339 player rows, 16 squads

v2 filtered to WC squads: 1248 rows across 36 squads

=== Combined v1+v2: 2901 rows, 47 squads ===
competition
UEFA Nations League    480
UEFA WCQ               470
AFC WCQ                409
CAF WCQ                382
Euro 2024              270
CONMEBOL WCQ 

In [16]:
# Check FBref's spelling for USA
print("Squad values containing 'unit' or 'state':")
for s in sorted(v2_all["Squad"].dropna().unique()):
    if "unit" in s.lower() or "state" in s.lower():
        print(f"  {s}")

# Add USA to the mapping and re-filter
WC_TO_FBREF_SQUAD["USA"] = "United States"

wc_players["fbref_squad"] = wc_players["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_players["team"])
wc_squads_fbref = set(wc_players["fbref_squad"].unique())

# Re-filter both v1 and v2 with the updated mapping
intl_v1 = wcq_all[wcq_all["Squad"].isin(wc_squads_fbref)].copy()
intl_v1["competition"] = intl_v1["confederation"].map({
    "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
    "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
})
intl_v2 = v2_all[v2_all["Squad"].isin(wc_squads_fbref)].copy()

intl_all = pd.concat([intl_v1, intl_v2], ignore_index=True)
print(f"\nCombined v1+v2: {len(intl_all)} rows, {intl_all['Squad'].nunique()} squads")

# Co-host coverage now
cohost_data = intl_all[intl_all["Squad"].isin(["United States", "Canada", "Mexico"])]
print("\nCo-host coverage:")
print(cohost_data.groupby(["Squad", "competition"]).size())

# Save final v1+v2 international stats
out = "../data/processed/international_stats_2026.csv"
intl_all.to_csv(out, index=False)
print(f"\nWrote {len(intl_all)} rows → {out}")


Squad values containing 'unit' or 'state':
  United States

Combined v1+v2: 2945 rows, 48 squads

Co-host coverage:
Squad          competition      
Canada         Copa America 2024    22
               Gold Cup 2025        23
Mexico         Copa America 2024    20
               Gold Cup 2025        23
United States  Copa America 2024    21
               Gold Cup 2025        23
dtype: int64

Wrote 2945 rows → ../data/processed/international_stats_2026.csv
